# M2 — Table cards + search (Qdrant)

**Goal of this step:** when someone asks a question, find the tables most likely to answer it.

**Analogy:** a library has thousands of books, so you search the **catalog cards** instead of reading every book.
Here, each of the 28 tables gets a short **card** that says what it holds, what one row means, how fresh it is,
and what it joins to. We then search the cards **by meaning**, not by exact words. That way "unique people" can
find a card that says "unique users", and "bot clicks" can find "suspicious traffic".

**How "search by meaning" works, in 3 lines**
1. An **embedding model** turns a sentence into a list of 384 numbers (a *vector*), like GPS coordinates for meaning.
2. Sentences that mean similar things get vectors that are close together.
3. **Qdrant** is a database built to find "the closest vectors" fast. The question's vector finds the closest card vectors.

**Files in this step**
| File | What it does |
|---|---|
| `catalog/table_notes.yaml` | Hand-written notes per table (description, grain, freshness, look-alikes, joins) |
| `catalog/make_cards.py` | Notes + facts from the database → one card per table in `catalog/cards/` |
| `catalog/load_qdrant.py` | Turns cards into vectors (FastEmbed, runs on your laptop) and stores them in Qdrant |
| `catalog/search.py` | `search_tables(question)` → the top tables (becomes an MCP tool in M6) |
| `catalog/check_search.py` | The done-test: 10 sample questions |

**Done when:** search returns sensible cards for 10 sample questions ✅

No API key is needed: the embedding model runs locally, and Qdrant is just a folder (`data/qdrant`).

In [1]:
import sys
sys.path.append("../catalog")   # so we can import the M2 scripts

import yaml
from make_cards import make_cards
from load_qdrant import build_index, embed, CARDS_DIR
from search import search_tables

/Users/anshikapuneet/Documents/Claude_history/capstone_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Make the cards
Each card = **my notes** (`table_notes.yaml`) + **facts read from the database**: columns, row count,
date range and 3 example rows.

In [2]:
make_cards()

  wrote dim_ad.yaml
  wrote dim_campaign.yaml
  wrote dim_advertiser.yaml
  wrote dim_user.yaml
  wrote dim_placement.yaml
  wrote dim_date.yaml
  wrote fact_ad_events.yaml
  wrote ref_suspicious_user_hours.yaml
  wrote agg_ad_hourly.yaml
  wrote agg_ad_daily.yaml
  wrote agg_ad_weekly.yaml
  wrote agg_campaign_hourly.yaml
  wrote agg_campaign_daily.yaml
  wrote agg_campaign_weekly.yaml
  wrote agg_advertiser_daily.yaml
  wrote agg_brand_daily.yaml
  wrote agg_category_daily.yaml
  wrote agg_placement_daily.yaml
  wrote agg_ad_daily_valid.yaml
  wrote agg_campaign_hourly_valid.yaml
  wrote agg_campaign_daily_valid.yaml
  wrote reach_campaign_daily.yaml
  wrote reach_campaign_total.yaml
  wrote reach_brand_daily.yaml
  wrote agg_campaign_daily_by_audience.yaml
  wrote live_ad_last6h.yaml
  wrote live_campaign_last6h.yaml
  wrote live_reach_campaign_last6h.yaml

28 cards written to /Users/anshikapuneet/Documents/Claude_history/capstone_project/catalog/cards


Here is one card. The `search_text` at the bottom is the part that gets searched.

In [3]:
print((CARDS_DIR / "reach_campaign_daily.yaml").read_text()[:1500])

table: reach_campaign_daily
description: Reach per campaign per day - number of unique users (people) who saw
  it and unique users who clicked. Do not add days together; use reach_campaign_total
  for multi-day reach.
grain: one row per campaign per day
freshness: final
traffic: all
joins:
  campaign_id: dim_campaign
lookalikes:
- reach_campaign_total
- agg_campaign_daily
- live_reach_campaign_last6h
columns:
- name: event_date
  type: DATE
- name: campaign_id
  type: BIGINT
- name: unique_users
  type: BIGINT
- name: unique_clickers
  type: BIGINT
row_count: 1601009
date_range: 2017-05-06 to 2017-05-12
examples:
- event_date: '2017-05-10'
  campaign_id: '296876'
  unique_users: '119'
  unique_clickers: '14'
- event_date: '2017-05-09'
  campaign_id: '46072'
  unique_users: '18'
  unique_clickers: '1'
- event_date: '2017-05-09'
  campaign_id: '285192'
  unique_users: '37'
  unique_clickers: '5'
search_text: 'Table reach_campaign_daily. Reach per campaign per day - number of unique
  us

## 2. What an embedding looks like
A sentence becomes 384 numbers. We compare two vectors with **cosine similarity**: 1.0 means the same meaning,
and close to 0 means unrelated. (The model loads on first use; the first run downloads about 130 MB.)

In [4]:
vector = embed(["unique people who saw the ad"])[0]
print("length:", len(vector))
print("first 5 numbers:", [round(x, 3) for x in vector[:5]])

length: 384
first 5 numbers: [-0.005, 0.05, -0.012, -0.025, 0.014]


In [5]:
def similarity(a, b):
    va, vb = embed([a, b])
    return round(sum(x * y for x, y in zip(va, vb)), 3)   # vectors are normalised, so this is the cosine

print("unique people  vs unique users      :", similarity("unique people", "unique users"))
print("unique people  vs CTR per hour      :", similarity("unique people", "CTR per hour"))
print("bot clicks     vs suspicious traffic:", similarity("bot clicks", "suspicious invalid traffic"))

unique people  vs unique users      : 0.827
unique people  vs CTR per hour      : 0.483
bot clicks     vs suspicious traffic: 0.597


## 3. Load the cards into Qdrant
`build_index()` embeds all 28 `search_text`s and stores them in the `table_cards` collection.
Re-running it rebuilds everything from scratch.

In [6]:
build_index()

Loaded 28 cards into Qdrant collection 'table_cards'


## 4. Search!
`search_tables(question, k)` returns the `k` closest tables with a score (higher = closer).

In [7]:
for table, score in search_tables("What was the CTR for campaign 359520 yesterday?"):
    print(f"{score:.3f}  {table}")

0.690  agg_campaign_daily_valid
0.688  agg_campaign_hourly_valid
0.688  live_campaign_last6h
0.686  agg_campaign_daily_by_audience
0.682  agg_campaign_daily


In [8]:
for table, score in search_tables("How many unique people saw campaign 83237 in the last 6 hours?"):
    print(f"{score:.3f}  {table}")

0.698  live_reach_campaign_last6h
0.679  live_campaign_last6h
0.667  reach_campaign_total
0.646  reach_campaign_daily
0.644  live_ad_last6h


## 5. The done-test: 10 sample questions
For each question, is the right table in the top 5? These are quick sanity checks, **not** the real test questions
(those are written separately in M3, so the system can't be tuned to them).

In [9]:
from check_search import run_checks
run_checks()

OK  rank=5  What was the CTR for campaign 359520 yesterday?
       expected agg_campaign_daily; got ['agg_campaign_daily_valid', 'agg_campaign_hourly_valid', 'live_campaign_last6h']
OK  rank=2  How many clicks did campaign 359520 get each hour today?
       expected agg_campaign_hourly; got ['live_campaign_last6h', 'agg_campaign_hourly', 'live_reach_campaign_last6h']
OK  rank=1  How many unique people saw campaign 83237 in the last 6 hours?
       expected live_reach_campaign_last6h; got ['live_reach_campaign_last6h', 'live_campaign_last6h', 'reach_campaign_total']
OK  rank=1  Total unique users reached by campaign 359520 over the whole week
       expected reach_campaign_total; got ['reach_campaign_total', 'reach_campaign_daily', 'agg_campaign_daily_by_audience']
OK  rank=2  Clicks for brand 95471 per day
       expected agg_brand_daily; got ['reach_brand_daily', 'agg_brand_daily', 'agg_advertiser_daily']
OK  rank=2  Campaign 359520 clicks with bot traffic removed
       expected agg_

## 6. What this tells us (important for the project!)
- ✅ **Done-test passed:** the right table is in the **top 5 for every question**.
- ⚠️ It's ranked **first only about half the time**. The winners are usually **look-alikes**. For example, "CTR
  yesterday" finds the `_valid` and hourly campaign tables before `agg_campaign_daily`, because they *sound* almost
  the same.

That's exactly the hypothesis in the design doc: **basic meaning-search can't tell look-alike tables apart.**
This is version **B (basic search)**. The next versions try to fix it:
- **C (M8):** keyword + meaning search, re-ranking and rewriting the question
- **D (M5):** the Neo4j table map adds connecting tables and lists look-alikes so the AI can choose
- **E (M7):** the agent reads cards, checks freshness and traffic, and retries

I didn't tweak the notes to make these 10 questions pass, because that would be cheating. The cards will be
reviewed once, then **frozen** before the M3 test questions are written.

**Your job before M3:** read `catalog/table_notes.yaml` and fix anything that's unclear. Then the cards are frozen.